In [ ]:
# =========================================================================
# 01 문제정의
# =========================================================================
# 예측 대상 : Item_Outlet_Sales (상품이 매장에서 얼마나 팔렸는가 = 매출액)
#
# 문제 종류 : 회귀(Regression)
#   회귀 = 연속된 숫자 예측 (매출액, 집값)  -> MAE, RMSE, R2
#   분류 = 정해진 범주 예측 (합격/불합격)   -> 정확도, F1
#
# 이 데이터의 구조적 한계
#   매출액 = Item_MRP(가격) x 판매수량
#            └ 있음           └ 없음!
#   수량을 설명할 변수(재고, 할인, 날짜, 날씨)가 하나도 없다
#   -> R2 상한이 0.6 안팎. 미리 알아야 나중에 헤매지 않는다

In [2]:
# =========================================================================
# 02 데이터가져오기 : 크기와 모양만 확인 (아직 분석 안 함)
# =========================================================================
import numpy as np
import pandas as pd

train = pd.read_csv('train.csv', low_memory=False)   # 타입 혼합 경고 방지
test  = pd.read_csv('test.csv',  low_memory=False)

print('train:', train.shape)   # (6818, 12)
print('test :', test.shape)    # (1705, 11)  <- 1개 적다

# test에 없는 열 = 우리가 맞춰야 할 정답
print('train에만 있는 열:', set(train.columns) - set(test.columns))

train: (6818, 12)
test : (1705, 11)
train에만 있는 열: {'Item_Outlet_Sales'}


In [3]:
# =========================================================================
# 03 EDA : 데이터를 '관찰'만 한다 (아직 고치지 않음)
#          목적 = "어디가 이상한지" 목록 만들기
# =========================================================================
train.info()
# object = 문자형 -> 숫자로 바꿔야 함 (04단계)
# Non-Null < 6818 -> 결측치 있음

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6818 entries, 0 to 6817
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            6818 non-null   object 
 1   Item_Weight                5656 non-null   float64
 2   Item_Fat_Content           6818 non-null   object 
 3   Item_Visibility            6818 non-null   float64
 4   Item_Type                  6818 non-null   object 
 5   Item_MRP                   6818 non-null   float64
 6   Outlet_Identifier          6818 non-null   object 
 7   Outlet_Establishment_Year  6818 non-null   int64  
 8   Outlet_Size                4878 non-null   object 
 9   Outlet_Location_Type       6818 non-null   object 
 10  Outlet_Type                6818 non-null   object 
 11  Item_Outlet_Sales          6818 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 639.3+ KB


In [ ]:
# --- 3-1. 결측치 확인 ---
print('=== train ===');  print(train.isnull().sum()[train.isnull().sum() > 0])
print('=== test  ===');  print(test.isnull().sum()[test.isnull().sum() > 0])

# [발견1] Item_Weight 1162개(17%), Outlet_Size 1940개(28%)
#         test에도 같은 열에 결측 -> 똑같이 처리해야 함

In [ ]:
# --- 3-2. 범주형 오염 찾기 ---
for col in train.select_dtypes('object').columns:
    print(f'{col:26s} 고유값 {train[col].nunique():5d}개')
# Item_Fat_Content 가 5개? 지방함량인데 5종류나? -> 의심하고 확인

print()
print(train['Item_Fat_Content'].value_counts())

# [발견2] 같은 의미인데 표기가 제각각
#   Low Fat / LF / low fat -> 같은 뜻
#   Regular / reg          -> 같은 뜻
#   컴퓨터는 '서로 다른 5개 값'으로 인식한다. 눈으로 봐야만 발견됨

In [ ]:
# --- 3-3. 0으로 위장한 결측치 ---
print('Item_Visibility 최솟값:', train['Item_Visibility'].min())
print('0인 행 개수      :', (train['Item_Visibility'] == 0).sum())

# [발견3] Item_Visibility = 진열대 노출 면적 비율
#         팔리는 상품인데 노출도 0? 물리적으로 불가능
#         실제로는 "모름"인데 0으로 기록된 것
#         isnull()로는 못 잡는다. 0이 말이 되는 값인지 '의미'를 따져야 함
#   ※ 나이 0살, 가격 0원, 키 0cm 보면 항상 의심할 것

In [ ]:
# --- 3-4. 정답과의 관계 ---
num = train.select_dtypes('number')
print(num.corr()['Item_Outlet_Sales'].sort_values(ascending=False))

# [발견4] 쓸모있는 열은 생각보다 적다
#   Item_MRP     0.567  <- 매우 유용
#   Item_Weight  0.021  <- 거의 없음 (결측은 17%나 되는데!)
#   ★ 열이 많다고 좋은 게 아니다

print()
for col in ['Outlet_Type', 'Outlet_Size', 'Outlet_Location_Type']:
    print(f'--- {col} ---')
    print(train.groupby(col)['Item_Outlet_Sales'].agg(['mean','count']).round(0))

# [발견5] Grocery Store 평균 매출이 압도적으로 낮다 (작은 가게니까)
#         숫자형이 아니라 상관계수 표엔 안 나왔지만 Item_MRP 다음으로 중요
#   ★ 상관계수만 믿으면 안 된다. 범주형은 따로 확인

In [4]:
# =========================================================================
# 04 데이터전처리 : 03에서 찾은 문제를 고친다
# =========================================================================
# 절대 규칙 3가지
#  1) train에 한 처리는 test에도 똑같이  -> for df in (train, test)
#  2) 채우는 '기준값'은 train에서만      -> test 정보 쓰면 데이터 누수
#  3) 문자는 숫자로 바꾼다               -> 모델은 글자를 이해 못 함

# --- 4-1. 표기 오염 정리 (발견2 해결) ---
fat_map = {'LF':'Low Fat', 'low fat':'Low Fat', 'reg':'Regular'}
for df in (train, test):
    df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_map)

print('정리 후:', train['Item_Fat_Content'].unique())   # 5종 -> 2종

정리 후: ['Low Fat' 'Regular']


In [5]:
# --- 4-2. Outlet_Size 결측 (발견1 해결) ---
# 최빈값(Medium) 대체는 부적절!
#   이 데이터의 결측은 랜덤이 아니라 '특정 매장이 통째로' 누락된 구조
#   실제 Small 매장을 Medium으로 채우면 거짓 정보 주입
#   'Unknown' 새 범주로 두면 모델이 "규모 정보 없는 그룹"으로 패턴을 찾는다
for df in (train, test):
    df['Outlet_Size'] = df['Outlet_Size'].fillna('Unknown')

print(train['Outlet_Size'].value_counts())

Outlet_Size
Medium     2228
Unknown    1940
Small      1925
High        725
Name: count, dtype: int64


In [6]:
# --- 4-3. Visibility 0 처리 (발견3 해결) ---
# 1) 0 -> NaN : 정직하게 "모름"으로 표시
# 2) 상품별 평균으로 채움 : 같은 상품은 어느 매장이든 진열면적이 비슷하므로
#                          전체 평균보다 훨씬 정확
for df in (train, test):
    df.loc[df['Item_Visibility'] == 0, 'Item_Visibility'] = np.nan

vis_mean = train.groupby('Item_Identifier')['Item_Visibility'].mean()   # 규칙2
for df in (train, test):
    df['Item_Visibility'] = df['Item_Visibility'].fillna(
        df['Item_Identifier'].map(vis_mean))
    df['Item_Visibility'] = df['Item_Visibility'].fillna(vis_mean.mean())

# --- 4-4. Item_Weight (상품코드마다 고정값 -> 같은 상품에서 가져오기) ---
wt_mean = train.groupby('Item_Identifier')['Item_Weight'].mean()
for df in (train, test):
    df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Identifier'].map(wt_mean))
    df['Item_Weight'] = df['Item_Weight'].fillna(wt_mean.mean())

print('결측:', train.isnull().sum().sum(), test.isnull().sum().sum())   # 0 0

결측: 0 0


In [7]:
# --- 4-5. 정답(y) 분리 ---
# ★★★ 이 셀을 건너뛰면 정답이 X에 남아 R2가 0.99가 나온다 (데이터 누수)
#      하지만 test엔 정답이 없으니 실전 예측은 완전 실패한다
y = train['Item_Outlet_Sales'].copy()
train_X = train.drop(columns=['Item_Outlet_Sales'])

print('y      :', y.shape)
print('train_X:', train_X.shape)    # (6818, 11) - test와 같은 열 수

y      : (6818,)
train_X: (6818, 11)


In [8]:
# --- 4-6. 원핫 인코딩 ---
# 원핫이란?
#   Outlet_Size      Size_High  Size_Medium  Size_Small
#      High     ->       1           0            0
#     Medium    ->       0           1            0
# 왜 High=1, Medium=2 로 번호 매기면 안 되나?
#   -> "Small(3)은 High(1)보다 3배 크다"는 없는 대소관계를 학습한다
# ★ train/test를 반드시 '합쳐서' 인코딩 (따로 하면 열 개수가 달라짐)
# Item_Identifier(1554종) 제외 - 원핫하면 열 1554개 폭발 -> 과적합

USE_COLS = ['Item_MRP', 'Outlet_Type', 'Outlet_Size']

both   = pd.get_dummies(pd.concat([train_X[USE_COLS], test[USE_COLS]], axis=0))
X      = both.iloc[:len(train_X)]
X_test = both.iloc[len(train_X):]

print('X     :', X.shape)
print('X_test:', X_test.shape, '<- 열 개수가 같아야 정상')

X     : (6818, 9)
X_test: (1705, 9) <- 열 개수가 같아야 정상


In [9]:
# --- 4-7. 누수 검사 ★ 반드시 실행 ---
def check_X(X, y, X_test, target='Item_Outlet_Sales'):
    assert target not in X.columns,       f'❌ 정답({target})이 X에 있습니다!'
    assert X.shape[1] == X_test.shape[1], '❌ train/test 열 개수 불일치'
    assert len(X) == len(y),              '❌ X와 y 행 수 불일치'
    top = X.corrwith(y).abs().sort_values(ascending=False)
    if top.iloc[0] > 0.95:
        print(f'⚠️  {top.index[0]} 상관 {top.iloc[0]:.3f} — 누수 의심')
    print('✅ 통과 |', X.shape[1], '열')

check_X(X, y, X_test)

✅ 통과 | 9 열


In [10]:
# =========================================================================
# 05 검증데이터 분할 및 학습
# =========================================================================
# 왜 또 나누나?
#   test엔 정답이 없다 -> 모델이 잘 만들어졌는지 확인 불가
#   -> train 일부를 떼어 '가짜 시험지'를 만든다
#
#   train 6818 ├ X_tr  5454 (80%) 공부용
#              └ X_val 1364 (20%) 시험용 (정답 있음)
#   test 1705                      진짜 시험 (정답 없음)

from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0    # ★ 고정 안 하면 실행할 때마다 성능이 달라진다
)
print('학습용:', X_tr.shape, '| 검증용:', X_val.shape)

학습용: (5454, 9) | 검증용: (1364, 9)


In [11]:
# --- 5-1. 기준 모델(baseline) 먼저 ---
# 바로 좋은 모델을 쓰지 않는다.
# 나중에 "정말 좋아진 게 맞나?" 비교할 기준선이 필요하기 때문
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(X_tr, y_tr)
pred_lr = lr.predict(X_val)
print('선형회귀 완료 | 절편:', round(lr.intercept_, 2))

선형회귀 완료 | 절편: -143.8


In [12]:
# --- 5-2. LightGBM ---
# 여러 트리를 순차로 만들며 앞 트리가 틀린 부분을 뒤 트리가 보완 (부스팅)
# 선형회귀와 달리 곡선 관계와 변수 간 상호작용을 잡아낸다
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    objective='regression',
    n_estimators=300,        # 트리 개수    (크면 과적합 위험)
    learning_rate=0.05,      # 반영 정도
    num_leaves=15,           # 트리 복잡도  (기본 31 -> 절반, 과적합 방지)
    min_child_samples=30,    # 잎 최소 샘플 (기본 20 -> 상향, 과적합 방지)
    random_state=0, verbose=-1,
).fit(X_tr, y_tr)

pred_lgbm = lgbm.predict(X_val)
print('LightGBM 완료')

LightGBM 완료


In [13]:
# =========================================================================
# 06 테스트 및 검증지표 확인
# =========================================================================
# MAE  : 평균적으로 얼마나 빗나갔나. 단위가 같아 가장 직관적
# RMSE : 큰 오차에 더 큰 벌점. MAE보다 훨씬 크면 = 가끔 크게 틀린다는 신호
# R2   : 변동을 몇 % 설명했나
#          0  = 평균으로 찍는 것과 동일 (무의미)
#         음수 = 평균으로 찍는 것보다 못하다
#         0.95 초과 -> ★ 기뻐하지 말고 누수를 의심할 것

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'[{name:10s}] MAE {mae:8.1f} | RMSE {rmse:8.1f} | R2 {r2_score(y_true,y_pred):.4f}')

evaluate('선형회귀',  y_val, pred_lr)
evaluate('LightGBM', y_val, pred_lgbm)

[선형회귀      ] MAE    798.8 | RMSE   1058.9 | R2 0.5681
[LightGBM  ] MAE    718.5 | RMSE   1022.7 | R2 0.5971


In [14]:
# --- 6-1. 과적합 검사 ★ 가장 중요 ---
# 검증 성능만 보면 안 된다. 반드시 '학습 성능과 함께' 본다
#   train 높고 val 낮다  -> 과적합 (기출문제 답만 외운 학생)
#   둘 다 낮다           -> 과소적합
#   둘 다 비슷하게 높다  -> 좋은 모델
for name, model in [('선형회귀', lr), ('LightGBM', lgbm)]:
    r2_tr  = r2_score(y_tr,  model.predict(X_tr))
    r2_val = r2_score(y_val, model.predict(X_val))
    gap = r2_tr - r2_val
    print(f'{name:10s} train {r2_tr:.4f} | val {r2_val:.4f} | 격차 {gap:.4f}'
          f'  {"과적합!" if gap > 0.1 else "양호"}')

# 판정 : 0.05 이하 양호 / 0.05~0.10 주의 / 0.10 초과 과적합
# 조치 : num_leaves 줄이기, n_estimators 줄이기,
#        min_child_samples 늘리기, 쓸모없는 열 빼기

선형회귀       train 0.5635 | val 0.5681 | 격차 -0.0045  양호
LightGBM   train 0.6442 | val 0.5971 | 격차 0.0471  양호
